In [ ]:
# ambiente local (celula 1 do notebook)

from dotenv import load_dotenv

load_dotenv("desenv.env", override=True)

from src.utils.gerenciador_sessao_spark_local import (
    GerenciadorSessaoSpark,
)

gerenciador_spark = GerenciadorSessaoSpark(
    nome_sessao="analise_transacoes",
    nome_arquivo_env_modelagem="desenv.env",
    exibir_configuracao=False,
)

spark = gerenciador_spark.criar_sessao_spark(
    db2=True
)

In [ ]:
# ambiente spark (celula 2 do notebook)

%run ./src/utils/gerenciador_sessao_spark_remoto.ipynb

In [ ]:
# ambiente spark (celula 3 do notebook)

%%spark

import os

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(
    env=env_spark
)

print("Conexão pronta para iniciar as queries.")

# MVP — transações correntes do cliente

A consulta usa exclusivamente `DB2GFP.TRAN_RLZD_INST_PCT`. Edite os três parâmetros na próxima célula antes de executar. `CODIGO_CLIENTE` é obrigatório e não possui valor padrão; as datas começam preenchidas com julho de 2026.

O intervalo é inclusivo, não pode ultrapassar 12 meses-calendário e deve estar integralmente nos 12 meses mais recentes em relação a `HOJE`. Somente transações efetivadas (`CD_EST_TRAN_INST = 0`) são retornadas.

%%spark

import os
from datetime import date, datetime

# Parâmetros editáveis do MVP.
CODIGO_CLIENTE = None
DATA_INICIO = "2026-07-01"
DATA_FIM = "2026-07-31"


def _falhar_contrato(codigo, mensagem, objeto):
    raise ErroContratoDados(
        codigo=codigo,
        mensagem=mensagem,
        objeto=objeto,
        acao="Corrigir os parâmetros da consulta antes de executar novamente.",
    )


def _normalizar_codigo_cliente(valor):
    if valor is None or isinstance(valor, bool):
        _falhar_contrato(
            "PARAMETRO_OBRIGATORIO",
            "CODIGO_CLIENTE deve ser informado como um número inteiro positivo.",
            "CODIGO_CLIENTE",
        )

    texto = str(valor).strip()
    if not texto:
        _falhar_contrato(
            "PARAMETRO_OBRIGATORIO",
            "CODIGO_CLIENTE deve ser informado como um número inteiro positivo.",
            "CODIGO_CLIENTE",
        )

    try:
        codigo_cliente = int(texto)
    except (TypeError, ValueError):
        _falhar_contrato(
            "PARAMETRO_INVALIDO",
            "CODIGO_CLIENTE deve ser um número inteiro positivo.",
            "CODIGO_CLIENTE",
        )

    if codigo_cliente <= 0:
        _falhar_contrato(
            "PARAMETRO_INVALIDO",
            "CODIGO_CLIENTE deve ser maior que zero.",
            "CODIGO_CLIENTE",
        )

    return codigo_cliente


def _normalizar_data(valor, nome_parametro):
    if valor is None or (isinstance(valor, str) and not valor.strip()):
        _falhar_contrato(
            "PARAMETRO_OBRIGATORIO",
            f"{nome_parametro} deve ser informada no formato AAAA-MM-DD.",
            nome_parametro,
        )

    if isinstance(valor, datetime):
        return valor.date()

    if isinstance(valor, date):
        return valor

    try:
        return date.fromisoformat(str(valor).strip())
    except (TypeError, ValueError):
        _falhar_contrato(
            "PARAMETRO_INVALIDO",
            f"{nome_parametro} deve ser uma data válida no formato AAAA-MM-DD.",
            nome_parametro,
        )


def _subtrair_doze_meses(valor):
    try:
        return valor.replace(year=valor.year - 1)
    except ValueError:
        # 29/02 passa a 28/02 no ano anterior.
        return valor.replace(year=valor.year - 1, day=28)


def validar_parametros_consulta(
    codigo_cliente,
    data_inicio,
    data_fim,
    data_referencia,
):
    codigo_cliente_final = _normalizar_codigo_cliente(codigo_cliente)
    data_inicio_final = _normalizar_data(data_inicio, "DATA_INICIO")
    data_fim_final = _normalizar_data(data_fim, "DATA_FIM")
    data_referencia_final = _normalizar_data(data_referencia, "HOJE")

    if data_inicio_final > data_fim_final:
        _falhar_contrato(
            "PERIODO_INVERTIDO",
            "DATA_INICIO deve ser menor ou igual a DATA_FIM.",
            "DATA_INICIO/DATA_FIM",
        )

    if data_inicio_final < _subtrair_doze_meses(data_fim_final):
        _falhar_contrato(
            "INTERVALO_EXCEDIDO",
            "O intervalo solicitado não pode ultrapassar 12 meses-calendário.",
            "DATA_INICIO/DATA_FIM",
        )

    limite_inferior = _subtrair_doze_meses(data_referencia_final)
    if (
        data_inicio_final < limite_inferior
        or data_fim_final < limite_inferior
        or data_inicio_final > data_referencia_final
        or data_fim_final > data_referencia_final
    ):
        _falhar_contrato(
            "PERIODO_FORA_DA_JANELA",
            (
                "DATA_INICIO e DATA_FIM devem estar entre "
                f"{limite_inferior.isoformat()} e "
                f"{data_referencia_final.isoformat()}, inclusive."
            ),
            "DATA_INICIO/DATA_FIM",
        )

    return {
        "codigo_cliente": codigo_cliente_final,
        "data_inicio": data_inicio_final,
        "data_fim": data_fim_final,
        "data_referencia": data_referencia_final,
    }


def montar_sql_transacoes(parametros):
    codigo_cliente = parametros["codigo_cliente"]
    data_inicio = parametros["data_inicio"].isoformat()
    data_fim = parametros["data_fim"].isoformat()

    return f"""
SELECT
    NR_TRAN_INST_PCT,
    CD_CLI,
    DT_TRAN,
    HR_TRAN,
    VL_TRAN,
    CD_TIP_MOE_CRR,
    CD_NTZ_CTB_TRAN,
    TX_DCR_TRAN,
    TX_CMPR_DCR_TRAN,
    TX_DCR_TRAN_OGNL,
    CD_TIP_TRAN,
    CD_INST_PCT,
    NR_MCA_PCT_OPB,
    NR_AG_TITR,
    CD_CT_TITR,
    DV_CT_TITR,
    CD_PRD,
    CD_CTGR_TRAN,
    CD_CTGR_TRAN_OGNL,
    IN_DVS_LCTO
FROM DB2GFP.TRAN_RLZD_INST_PCT
WHERE CD_CLI = {codigo_cliente}
  AND DT_TRAN BETWEEN DATE('{data_inicio}') AND DATE('{data_fim}')
  AND CD_EST_TRAN_INST = 0
""".strip()


%%spark

etapa_atual = "consultar_transacoes_correntes"

try:
    parametros_consulta = validar_parametros_consulta(
        codigo_cliente=CODIGO_CLIENTE,
        data_inicio=DATA_INICIO,
        data_fim=DATA_FIM,
        data_referencia=os.environ.get("HOJE"),
    )

    sql_transacoes = montar_sql_transacoes(parametros_consulta)

    df_transacoes = (
        cliente_db2.run_select(
            sql=sql_transacoes,
            fetchsize=1000,
            query_timeout=300,
        )
        .orderBy(
            col("DT_TRAN").asc(),
            col("HR_TRAN").asc_nulls_last(),
            col("NR_TRAN_INST_PCT").asc(),
        )
    )

    print(
        "Consulta preparada com sucesso: "
        f"cliente={parametros_consulta['codigo_cliente']}, "
        f"período={parametros_consulta['data_inicio']} a "
        f"{parametros_consulta['data_fim']}."
    )
    print("Resultado disponível no DataFrame df_transacoes.")
    df_transacoes.show(n=100, truncate=False)

except Exception as exc:
    registrar_erro_rotina(etapa_atual, exc)
    raise


# Estudo de validação


In [ ]:
%%spark
from datetime import date, datetime
CODIGO_CLIENTE = None
DATA_INICIO = '2026-07-01'
DATA_FIM = '2026-07-31'
if CODIGO_CLIENTE is None or DATA_INICIO is None or DATA_FIM is None: raise ValueError('Parâmetros obrigatórios ausentes.')
inicio=datetime.strptime(DATA_INICIO,'%Y-%m-%d').date(); fim=datetime.strptime(DATA_FIM,'%Y-%m-%d').date(); hoje=date.today()
if inicio>fim or (fim.year-inicio.year)*12+fim.month-inicio.month>12: raise ValueError('Intervalo inválido.')
if inicio<hoje.replace(year=hoje.year-1) or fim>hoje: raise ValueError('Período fora da janela móvel.')
print('Parâmetros válidos; CODIGO_CLIENTE é usado somente no filtro interno.')

## 1. Tabela principal
Começamos exclusivamente por DB2GFP.TRAN_RLZD_INST_PCT para confirmar moedas, valores, crédito/débito, instituição, marca, produto, tipo e categorias.

In [ ]:
%%spark
def consultar(sql): return cliente_db2.run_select(sql=sql, fetchsize=1000, query_timeout=300)
base=f'''FROM DB2GFP.TRAN_RLZD_INST_PCT WHERE CD_CLI={CODIGO_CLIENTE} AND DT_TRAN BETWEEN DATE('{DATA_INICIO}') AND DATE('{DATA_FIM}') AND CD_EST_TRAN_INST=0'''
df_principal=consultar(f'''SELECT CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL,SUM(CASE WHEN CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='C' THEN VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='D' THEN VL_TRAN ELSE 0 END) VL_DEBITOS {base} GROUP BY CD_TIP_MOE_CRR''')
df_instituicoes=consultar(f'''SELECT CD_INST_PCT,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL {base} GROUP BY CD_INST_PCT,CD_TIP_MOE_CRR''')
df_marcas=consultar(f'''SELECT NR_MCA_PCT_OPB,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL {base} GROUP BY NR_MCA_PCT_OPB,CD_TIP_MOE_CRR''')
df_produtos=consultar(f'''SELECT CD_PRD,CD_TIP_TRAN,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL,SUM(CASE WHEN NR_SEQL_CT_CLI IS NULL THEN 1 ELSE 0 END) QT_SEM_SEQUENCIA {base} GROUP BY CD_PRD,CD_TIP_TRAN,CD_TIP_MOE_CRR''')
df_integridade=consultar(f'''SELECT COUNT(*) QT_REGISTROS,COUNT(DISTINCT NR_TRAN_INST_PCT) QT_TRANSACOES,MIN(DT_TRAN) DT_MIN,MAX(DT_TRAN) DT_MAX,SUM(CASE WHEN CD_CTGR_TRAN IS NULL THEN 1 ELSE 0 END) QT_SEM_CATEGORIA {base}''')
for nome,frame in [('Resumo por moeda',df_principal),('Instituições',df_instituicoes),('Marcas',df_marcas),('Produtos e tipos',df_produtos),('Integridade',df_integridade)]: print('\n### '+nome); frame.show(truncate=False)
print('CONFIRMADO NOS DADOS quando campos e códigos observados forem consistentes; caso contrário, CONFIRMADO COM LIMITAÇÃO.')

## 2. Lacuna concreta — contas Open Finance
A tabela principal não possui o universo de contas nem estados de autorização. INF_OPB_CT_CLI entra somente para essa lacuna. Conta autorizada exige NR_SEQL_AUTZ_OPB preenchido e IN_AUTZ_EXB_TRAN='S'.

In [ ]:
%%spark
df_contas=consultar(f'''SELECT CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CD_MDLD_PRD,TRIM(IN_AUTZ_EXB_TRAN) IN_AUTZ_EXB_TRAN,TRIM(IN_EXB_DADO_CT) IN_EXB_DADO_CT,CD_EST_RCS_OPB,COUNT(*) QT_RELACIONAMENTOS FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CD_MDLD_PRD,TRIM(IN_AUTZ_EXB_TRAN),TRIM(IN_EXB_DADO_CT),CD_EST_RCS_OPB''')
df_flags=consultar(f'''SELECT TRIM(IN_AUTZ_EXB_TRAN) IN_AUTZ_EXB_TRAN,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY TRIM(IN_AUTZ_EXB_TRAN)''')
df_chaves=consultar(f'''SELECT NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY NR_MCA_PCT_OPB,NR_SEQL_CT_CLI HAVING COUNT(*)>1''')
df_contas.show(truncate=False); df_flags.show(truncate=False); print('Chaves duplicadas:',df_chaves.count())
print('Classificação: CONFIRMADO NOS DADOS se valores S/N e chaves forem únicos; caso contrário, NÃO CONFIRMADO.')

In [ ]:
%%spark
cte=f'''WITH R AS (SELECT CD_CLI_TITR_CT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CASE WHEN NR_SEQL_AUTZ_OPB IS NOT NULL AND TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 1 ELSE 0 END AUTORIZADA,ROW_NUMBER() OVER(ORDER BY CD_INST_PCT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI) CONTA_ESTUDO FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE}),B AS (SELECT NR_TRAN_INST_PCT,CD_CLI,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,CD_TIP_MOE_CRR,VL_TRAN,CD_NTZ_CTB_TRAN,CD_PRD,CD_TIP_TRAN,CD_CTGR_TRAN FROM DB2GFP.TRAN_RLZD_INST_PCT WHERE CD_CLI={CODIGO_CLIENTE} AND DT_TRAN BETWEEN DATE('{DATA_INICIO}') AND DATE('{DATA_FIM}') AND CD_EST_TRAN_INST=0)'''
classificacao='CASE WHEN b.NR_SEQL_CT_CLI IS NULL THEN \'SEM_SEQUENCIA\' WHEN r.CD_CLI_TITR_CT IS NULL THEN \'SEM_RELACIONAMENTO\' WHEN r.AUTORIZADA=1 THEN \'CONTA_AUTORIZADA\' ELSE \'CONTA_NAO_AUTORIZADA\' END'
df_associacao=consultar(cte+f'''SELECT b.CD_TIP_MOE_CRR,{classificacao} CLASSIFICACAO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM B b LEFT JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI GROUP BY b.CD_TIP_MOE_CRR,{classificacao}''')
df_linhas=consultar(cte+'''SELECT COUNT(*) QT_LINHAS,COUNT(DISTINCT b.NR_TRAN_INST_PCT) QT_TRANSACOES FROM B b LEFT JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI''')
df_contas_sem_mov=consultar(cte+'''SELECT COUNT(*) QT_CONTAS_AUTORIZADAS_SEM_MOV FROM (SELECT DISTINCT CONTA_ESTUDO,CD_CLI_TITR_CT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI FROM R WHERE AUTORIZADA=1) r LEFT JOIN (SELECT DISTINCT CD_CLI,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI FROM B) b ON b.CD_CLI=r.CD_CLI_TITR_CT AND b.NR_MCA_PCT_OPB=r.NR_MCA_PCT_OPB AND b.NR_SEQL_CT_CLI=r.NR_SEQL_CT_CLI WHERE b.CD_CLI IS NULL''')
df_associacao.show(truncate=False); df_linhas.show(truncate=False); df_contas_sem_mov.show(truncate=False)
print('CONFIRMADO NOS DADOS se QT_LINHAS=QT_TRANSACOES, chaves únicas e cobertura quantificada; caso contrário, CONFIRMADO COM LIMITAÇÃO ou NÃO CONFIRMADO.')

## 3. Resumo por conta
Contas autorizadas recebem rótulos temporários não identificadores. Totais, créditos e débitos ficam separados por moeda.

In [ ]:
%%spark
df_contas_resumo=consultar(cte+'''SELECT r.CONTA_ESTUDO,r.CD_INST_PCT,r.CD_TIP_CT_OPB,r.CD_PRD,b.CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM B b JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 GROUP BY r.CONTA_ESTUDO,r.CD_INST_PCT,r.CD_TIP_CT_OPB,r.CD_PRD,b.CD_TIP_MOE_CRR''')
df_contas_tipos=consultar(cte+'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,b.CD_PRD,b.CD_TIP_TRAN,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL FROM B b JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,b.CD_PRD,b.CD_TIP_TRAN''')
df_contas_resumo.show(truncate=False); df_contas_tipos.show(truncate=False)
print('CONFIRMADO NOS DADOS quando a associação autorizada estiver segura; contas sem movimento permanecem no estudo de cobertura.')

## 4. Grupos e categorias por conta
A categoria atual é a referência. Categorias originais permanecem separadas; códigos sem correspondência ficam em buckets explícitos.

In [ ]:
%%spark
df_cat_dups=consultar('SELECT CD_CTGR_TRAN,COUNT(*) QT FROM DB2GFP.CTGR_TRAN_OPB GROUP BY CD_CTGR_TRAN HAVING COUNT(*)>1')
df_group_dups=consultar('SELECT CD_GR_CTGR_TRAN,COUNT(*) QT FROM DB2GFP.GR_CTGR_TRAN GROUP BY CD_GR_CTGR_TRAN HAVING COUNT(*)>1')
dominios_unicos=(df_cat_dups.count()==0 and df_group_dups.count()==0)
print('Domínios sem duplicidade:',dominios_unicos)
if not dominios_unicos: raise ValueError('NÃO CONFIRMADO: domínio de categoria ou grupo duplicado; sumarização interpretada bloqueada.')
df_grupos=consultar(cte+'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,COALESCE(g.CD_GR_CTGR_TRAN,'SEM_GRUPO') CD_GRUPO,COALESCE(g.TX_DCR_GR_CTGR,'Sem grupo') DS_GRUPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL FROM B b JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 LEFT JOIN DB2GFP.CTGR_TRAN_OPB c ON c.CD_CTGR_TRAN=b.CD_CTGR_TRAN LEFT JOIN DB2GFP.GR_CTGR_TRAN g ON g.CD_GR_CTGR_TRAN=c.CD_GR_CTGR_TRAN GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,g.CD_GR_CTGR_TRAN,g.TX_DCR_GR_CTGR''')
df_categorias=consultar(cte+'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,COALESCE(c.CD_CTGR_TRAN,'SEM_CATEGORIA') CD_CATEGORIA,COALESCE(c.TX_DCR_CTGR_TRAN,'Sem categoria') DS_CATEGORIA,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL FROM B b JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 LEFT JOIN DB2GFP.CTGR_TRAN_OPB c ON c.CD_CTGR_TRAN=b.CD_CTGR_TRAN GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,c.CD_CTGR_TRAN,c.TX_DCR_CTGR_TRAN''')
df_grupos.show(truncate=False); df_categorias.show(truncate=False)
print('CONFIRMADO NOS DADOS se domínios únicos e cobertura fecharem; duplicidades tornam a interpretação NÃO CONFIRMADA.')

## 5. Reconciliação e conclusão
Por moeda, cliente = contas autorizadas + classes não associadas. Grupos/categorias conhecidos + buckets sem classificação devem representar todas as transações autorizadas.

In [ ]:
%%spark
df_reconciliacao=consultar(cte+'''SELECT b.CD_TIP_MOE_CRR,'CLIENTE' ESCOPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL FROM B b GROUP BY b.CD_TIP_MOE_CRR UNION ALL SELECT b.CD_TIP_MOE_CRR,CASE WHEN r.AUTORIZADA=1 THEN 'CONTAS_AUTORIZADAS' ELSE 'NAO_ASSOCIADA' END,COUNT(*),SUM(b.VL_TRAN) FROM B b LEFT JOIN R r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI GROUP BY b.CD_TIP_MOE_CRR,CASE WHEN r.AUTORIZADA=1 THEN 'CONTAS_AUTORIZADAS' ELSE 'NAO_ASSOCIADA' END''')
df_reconciliacao.show(truncate=False)
print('CONFIRMADO NOS DADOS se os totais por moeda fecharem; diferenças ficam explicadas por não associação ou marcadas como inesperadas.')
print('Menor conjunto: TRAN_RLZD_INST_PCT; INF_OPB_CT_CLI para contas; CTGR_TRAN_OPB e GR_CTGR_TRAN somente para descrições únicas.')
print('Nenhuma fonte externa de autorização participa do estudo ou da solução.')

In [ ]:
%%spark
df_estados=consultar(f'''SELECT CASE WHEN NR_SEQL_AUTZ_OPB IS NULL THEN 'SEM_RELACIONAMENTO_OF' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND CD_EST_RCS_OPB=2 THEN 'AUTORIZADA_DISPONIVEL' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND TRIM(IN_EXB_DADO_CT)='N' THEN 'AUTORIZADA_OCULTA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 'AUTORIZADA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='N' THEN 'NAO_AUTORIZADA' ELSE 'ESTADO_NAO_RECONHECIDO' END STATUS_CONTA,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY CASE WHEN NR_SEQL_AUTZ_OPB IS NULL THEN 'SEM_RELACIONAMENTO_OF' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND CD_EST_RCS_OPB=2 THEN 'AUTORIZADA_DISPONIVEL' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND TRIM(IN_EXB_DADO_CT)='N' THEN 'AUTORIZADA_OCULTA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 'AUTORIZADA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='N' THEN 'NAO_AUTORIZADA' ELSE 'ESTADO_NAO_RECONHECIDO' END''')
df_estados.show(truncate=False); print('Estados são atributos observados; não reconstruímos histórico de consentimento.')